In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

# 1. Load the source dataset
raw = pd.read_csv("../../data/raw/movies_raw_enriched.csv")

# 2. Keep required content and display metadata
df = raw[
    ["movie_id", "title", "overview", "genres", "keywords",
     "cast", "director", "release_date", "poster_url"]
].copy()

# 3. Validate identifiers and titles
df = df.dropna(subset=["movie_id", "title"]).copy()
df["title"] = df["title"].astype(str).str.strip()
df = df[df["title"] != ""].reset_index(drop=True)

if df["movie_id"].duplicated().any():
    raise ValueError("movie_id must uniquely identify each movie.")

# 4. Preserve movies with partial metadata
text_columns = ["overview", "genres", "keywords", "cast", "director"]
df[text_columns] = df[text_columns].fillna("").astype(str)

# 5. Parse JSON-encoded genres and keywords
def parse_list(value):
    try:
        result = json.loads(value)
        return result if isinstance(result, list) else []
    except (json.JSONDecodeError, TypeError):
        return []

# 6. Build one text representation per movie
def build_text(row):
    genres = " ".join(parse_list(row["genres"]))
    keywords = " ".join(parse_list(row["keywords"]))
    cast = " ".join(row["cast"].split("|")[:5])
    director = " ".join(row["director"].split("|"))

    return " ".join(
        part for part in [
            row["overview"], genres, keywords, cast, director
        ] if part.strip()
    )

df["release_year"] = pd.to_datetime(
    df["release_date"], errors="coerce"
).dt.year

df["text"] = df.apply(build_text, axis=1)

# 7. Exclude records with no usable textual content
df = df[df["text"].str.strip().ne("")].reset_index(drop=True)

# 8. Convert movie text into TF-IDF vectors
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(df["text"])

print("Movies indexed:", len(df))
print("TF-IDF matrix:", tfidf_matrix.shape)

# 9. Recommend by content similarity, with explicit duplicate-title handling
def recommend(title, n=5, movie_id=None):
    matches = np.flatnonzero(
        df["title"].str.casefold().eq(title.strip().casefold()).to_numpy()
    )

    if len(matches) == 0:
        return f"No movie found with title: {title}"

    if movie_id is None and len(matches) > 1:
        return df.iloc[matches][
            ["movie_id", "title", "release_year", "poster_url"]
        ].reset_index(drop=True)

    if movie_id is not None:
        selected = [
            idx for idx in matches
            if df.iloc[idx]["movie_id"] == movie_id
        ]
        if not selected:
            return f"Movie ID {movie_id} does not match '{title}'."
        query_idx = selected[0]
    else:
        query_idx = matches[0]

    scores = linear_kernel(
        tfidf_matrix[query_idx], tfidf_matrix
    ).ravel()

    ranked = np.argsort(-scores)
    ranked = [idx for idx in ranked if idx != query_idx][:max(0, n)]

    result = df.iloc[ranked][
        ["movie_id", "title", "release_year", "poster_url"]
    ].copy()
    result["similarity"] = scores[ranked]

    return result.reset_index(drop=True)

# 10. Smoke tests
for title in ["The Notebook", "Star Wars", "The Dark Knight", "Toy Story"]:
    print(f"\n=== {title} ===")
    print(recommend(title).to_string(index=False))

Movies indexed: 34989
TF-IDF matrix: (34989, 109119)

=== The Notebook ===
 movie_id             title  release_year                                                     poster_url  similarity
    12650           Il Mare        2000.0 https://image.tmdb.org/t/p/w500csJ2BCbQaaNL4n8dk3W91QiAWmV.jpg    0.159073
   262355   Thamarai Nenjam        1968.0                                                            NaN    0.123684
     4588     Lust, Caution        2007.0 https://image.tmdb.org/t/p/w5006c1tqfJEBuIyhQC19SLlLQAUAvJ.jpg    0.119351
   640306  Interrupted Song        1960.0 https://image.tmdb.org/t/p/w500b4rrec9s6iL73evAa8KzUnaIGRG.jpg    0.116880
    66336 Apoorva Raagangal        1975.0 https://image.tmdb.org/t/p/w50060gSB8RinZAhrunuhJFvvCprmQ8.jpg    0.113667

=== Star Wars ===
 movie_id                                        title  release_year                                                     poster_url  similarity
     1891                      The Empire Strikes Back      

In [2]:
test_movies = [
    "The Notebook",       # Romance / drama
    "The Dark Knight",   # Superhero / action
    "Toy Story",          # Animation / family
    "The Godfather",      # Crime / drama
    "Saiyaara",           # Indian romance / musical
    "Drishyam",           # Indian thriller
    "Star Wars",          # Science fiction
]

for title in test_movies:
    print(f"\n{'=' * 60}")
    print(f"Query movie: {title}")
    print("=" * 60)
    print(recommend(title, n=5))


Query movie: The Notebook
   movie_id              title  release_year  \
0     12650            Il Mare        2000.0   
1    262355    Thamarai Nenjam        1968.0   
2      4588      Lust, Caution        2007.0   
3    640306   Interrupted Song        1960.0   
4     66336  Apoorva Raagangal        1975.0   

                                          poster_url  similarity  
0  https://image.tmdb.org/t/p/w500csJ2BCbQaaNL4n8...    0.159073  
1                                                NaN    0.123684  
2  https://image.tmdb.org/t/p/w5006c1tqfJEBuIyhQC...    0.119351  
3  https://image.tmdb.org/t/p/w500b4rrec9s6iL73ev...    0.116880  
4  https://image.tmdb.org/t/p/w50060gSB8RinZAhrun...    0.113667  

Query movie: The Dark Knight
   movie_id                  title  release_year  \
0     49026  The Dark Knight Rises        2012.0   
1       272          Batman Begins        2005.0   
2       268                 Batman        1989.0   
3       414         Batman Forever        19

In [3]:

# Fixed evaluation set: keep these queries unchanged for future comparisons.
evaluation_queries = [
    ("The Notebook", None),
    ("The Dark Knight", None),
    ("Toy Story", None),
    ("The Godfather", None),
    ("Saiyaara", None),
    ("Drishyam", 244049),  # Explicitly select the 2013 version.
    ("Star Wars", None),
]

evaluation_rows = []

for title, movie_id in evaluation_queries:
    results = recommend(title, n=5, movie_id=movie_id)

    if not isinstance(results, pd.DataFrame):
        print(f"Could not evaluate {title}:")
        print(results)
        continue

    for rank, (_, movie) in enumerate(results.iterrows(), start=1):
        evaluation_rows.append({
            "query_movie": title,
            "query_movie_id": movie_id,
            "rank": rank,
            "recommended_movie_id": movie["movie_id"],
            "recommended_title": movie["title"],
            "release_year": movie["release_year"],
            "similarity": movie["similarity"],
            "relevance": pd.NA,
        })

evaluation_set = pd.DataFrame(evaluation_rows)

print("Evaluation recommendations:", len(evaluation_set))
display(evaluation_set)


Evaluation recommendations: 35


,query_movie,query_movie_id,rank,recommended_movie_id,recommended_title,release_year,similarity,relevance
0,The Notebook,NaN,1,12650,Il Mare,2000.0,0.159073,<NA>
1,The Notebook,NaN,2,262355,Thamarai Nenjam,1968.0,0.123684,<NA>
2,The Notebook,NaN,3,4588,"Lust, Caution",2007.0,0.119351,<NA>
3,The Notebook,NaN,4,640306,Interrupted Song,1960.0,0.116880,<NA>
4,The Notebook,NaN,5,66336,Apoorva Raagangal,1975.0,0.113667,<NA>
5,The Dark Knight,NaN,1,49026,The Dark Knight Rises,2012.0,0.365108,<NA>
6,The Dark Knight,NaN,2,272,Batman Begins,2005.0,0.325960,<NA>
7,The Dark Knight,NaN,3,268,Batman,1989.0,0.309641,<NA>
8,The Dark Knight,NaN,4,414,Batman Forever,1995.0,0.254873,<NA>
9,The Dark Knight,NaN,5,414906,The Batman,2022.0,0.202981,<NA>


In [4]:

for title in ["The Notebook", "Saiyaara"]:
    matches = df[df["title"].str.casefold() == title.casefold()]

    print(f"\n{'=' * 60}\n{title} — SOURCE METADATA\n{'=' * 60}")

    if matches.empty:
        print("Movie not found.")
        continue

    for _, movie in matches.iterrows():
        for field in [
            "movie_id", "title", "release_year",
            "overview", "genres", "keywords", "cast", "director"
        ]:
            print(f"\n{field}:")
            print(movie[field])

        print("\nTop 5 recommendations:")
        print(recommend(title, n=5, movie_id=movie["movie_id"])[
            ["title", "similarity"]
        ].to_string(index=False))



The Notebook — SOURCE METADATA

movie_id:
11036

title:
The Notebook

release_year:
2004.0

overview:
An epic love story centered around an older man who reads aloud to a woman with Alzheimer's. From a faded notebook, the old man's words bring to life the story about a couple who is separated by World War II, and is then passionately reunited, seven years later, after they have taken different paths.

genres:
["Romance", "Drama"]

keywords:
["secret love", "soulmates", "based on novel or book", "love of one's life", "fight", "poem", "river", "sadness", "class", "dementia", "tears", "candle", "mailbox", "valentine's day", "romantic"]

cast:
Ryan Gosling | Rachel McAdams | Gena Rowlands | James Garner | Joan Allen

director:
Nick Cassavetes

Top 5 recommendations:
            title  similarity
          Il Mare    0.159073
  Thamarai Nenjam    0.123684
    Lust, Caution    0.119351
 Interrupted Song    0.116880
Apoorva Raagangal    0.113667

Saiyaara — SOURCE METADATA

movie_id:
1241634

In [5]:
for title in ["The Notebook", "Saiyaara"]:
    idx = df.index[df["title"].str.casefold() == title.casefold()][0]
    row_vector = tfidf_matrix[idx]
    feature_names = tfidf.get_feature_names_out()

    weights = row_vector.toarray().ravel()
    top_indices = weights.argsort()[::-1][:20]

    print(f"\n=== {title}: TOP TF-IDF FEATURES ===")
    for i in top_indices:
        if weights[i] > 0:
            print(f"{feature_names[i]:<25} {weights[i]:.4f}")


=== The Notebook: TOP TF-IDF FEATURES ===
rowlands                  0.1979
mailbox                   0.1905
gena                      0.1811
cassavetes                0.1811
mcadams                   0.1777
gosling                   0.1777
aloud                     0.1777
candle                    0.1703
soulmates                 0.1683
faded                     0.1666
passionately              0.1650
garner                    0.1635
notebook                  0.1635
love                      0.1617
dementia                  0.1597
reads                     0.1556
valentine                 0.1523
alzheimer                 0.1515
allen                     0.1415
sadness                   0.1411

=== Saiyaara: TOP TF-IDF FEATURES ===
ahaan                     0.2566
padda                     0.2469
aneet                     0.2469
undeniable                0.2267
deepen                    0.2267
agrawal                   0.2160
contrasting               0.2070
mohit                     0

In [6]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

# Temporary experiment: exclude cast and director.
experiment_df = df.copy()

def build_plot_text(row):
    import json

    def parse(value):
        try:
            result = json.loads(value)
            return " ".join(result) if isinstance(result, list) else ""
        except (json.JSONDecodeError, TypeError):
            return ""

    return " ".join([
        str(row["overview"]),
        parse(row["genres"]),
        parse(row["keywords"]),
    ])

experiment_df["plot_text"] = experiment_df.apply(build_plot_text, axis=1)

experiment_tfidf = TfidfVectorizer(stop_words="english")
experiment_matrix = experiment_tfidf.fit_transform(
    experiment_df["plot_text"]
)

def recommend_experiment(title, n=5, movie_id=None):
    matches = experiment_df.index[
        experiment_df["title"].str.casefold() == title.casefold()
    ].tolist()

    if movie_id is not None:
        matches = [
            i for i in matches
            if experiment_df.loc[i, "movie_id"] == movie_id
        ]

    if not matches:
        return f"Movie not found: {title}"

    query_idx = matches[0]
    scores = linear_kernel(
        experiment_matrix[query_idx], experiment_matrix
    ).ravel()

    ranked = scores.argsort()[::-1]
    ranked = [i for i in ranked if i != query_idx][:n]

    result = experiment_df.iloc[ranked][
        ["movie_id", "title", "release_year"]
    ].copy()
    result["similarity"] = scores[ranked]
    return result.reset_index(drop=True)

for title, movie_id in [
    ("The Notebook", 11036),
    ("Saiyaara", 1241634),
]:
    print(f"\n=== {title}: PLOT + GENRES + KEYWORDS ===")
    print(recommend_experiment(
        title, n=5, movie_id=movie_id
    ).to_string(index=False))



=== The Notebook: PLOT + GENRES + KEYWORDS ===
 movie_id               title  release_year  similarity
   640306    Interrupted Song        1960.0    0.280650
   729944      The Stepmother        1971.0    0.272109
    12650             Il Mare        2000.0    0.250859
   939410   Chalo Ishq Larain        2002.0    0.206374
   304337 Pallavi Anu Pallavi        1983.0    0.188629

=== Saiyaara: PLOT + GENRES + KEYWORDS ===
 movie_id                 title  release_year  similarity
   594675                 Qaidi        1962.0    0.390734
  1017636         Swara Kalpana        1989.0    0.383084
   943592        Pa Ma Ga Re Sa        2009.0    0.381025
  1768993                   西厢记        1965.0    0.381025
  1389390 Firecracker of Tehran        1961.0    0.381025


In [7]:

import json
import pickle
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# Paths
ROOT = Path(__file__).resolve().parents[1]
DATA_PATH = ROOT / "data" / "raw" / "movies_raw_enriched.csv"
MODEL_PATH = ROOT / "backend" / "movie_recommender.pkl"

# Load the required columns
columns = [
    "movie_id",
    "title",
    "overview",
    "genres",
    "keywords",
    "cast",
    "director",
]

df = pd.read_csv(DATA_PATH)[columns]

# Remove rows with missing values in any selected column
df = df.dropna(subset=columns).reset_index(drop=True)

# Convert JSON lists into readable text
def parse_list(value):
    try:
        result = json.loads(value)
        return " ".join(result) if isinstance(result, list) else ""
    except (json.JSONDecodeError, TypeError):
        return ""

# Combine movie information into one text field
def build_text(row):
    genres = parse_list(row["genres"])
    keywords = parse_list(row["keywords"])
    cast = row["cast"].replace("|", " ")
    director = row["director"].replace("|", " ")

    return " ".join([
        row["overview"],
        genres,
        keywords,
        cast,
        director,
    ])

df["text"] = df.apply(build_text, axis=1)

# Convert text into TF-IDF vectors
vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(df["text"])

# Save the model and movie data
with open(MODEL_PATH, "wb") as file:
    pickle.dump(
        {
            "movies": df.drop(columns=["text"]),
            "vectorizer": vectorizer,
            "tfidf_matrix": tfidf_matrix,
        },
        file,
    )

print("Movies used:", len(df))
print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Model saved to:", MODEL_PATH)


NameError: name '__file__' is not defined